In [1]:
#FORESIGHT - Week 1: Data Pipeline
# Ingest sale.csv, sku.csv, calendar.csv, inventory.csv and produces an analysis-ready dataset.

In [2]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("foresight")

dq_log = [] # running list of data quality findings -> become a report

def note(message):
    log.info(message)
    dq_log.append(message)

In [3]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILES = {
    "sales": RAW_DIR / "sale.csv",
    "sku": RAW_DIR / "sku.csv",
    "calendar": RAW_DIR / "calendar.csv",
    "inventory": RAW_DIR / "inventory.csv",
}

SKU_ID_WIDTH = 5    # sale.csv has 4-digit SKU ids, others have 5-digit
STORE_ID_WIDTH = 2  # sale.csv has 3-digit store ids, inventory.csv has 2-digit

In [4]:
raw = {}
for name, path in RAW_FILES.items():
    df = pd.read_csv(path)
    note(f"message={name} : shape ={len(df):,} rows, {len(df.columns)} columns from {path.name}")
    raw[name] = df

for name, df in raw.items():
    print(f"\n=== {name} ===")
    print(df.head(3))
    

INFO | message=sales : shape =500,000 rows, 16 columns from sale.csv
INFO | message=sku : shape =5,000 rows, 7 columns from sku.csv
INFO | message=calendar : shape =1,765 rows, 10 columns from calendar.csv
INFO | message=inventory : shape =26,408 rows, 6 columns from inventory.csv



=== sales ===
         date    receipt_id store_id   sku_id  customer_quantity  unit_price  \
0  2024-01-01  RCPT00000001    ST007  SKU0033                  4       51.35   
1  2024-01-01  RCPT00000002    ST004  SKU0174                  9      743.67   
2  2024-01-01  RCPT00000003    ST013  SKU0052                  2       98.04   

   total_value   channel discount_promo_id  month  year  quarter  week  \
0       205.40  In-Store               NaN      1  2024        1     1   
1      6693.03    Online               NaN      1  2024        1     1   
2       196.08  In-Store               NaN      1  2024        1     1   

   day_of_week month_name day_name  
0            0    January   Monday  
1            0    January   Monday  
2            0    January   Monday  

=== sku ===
     sku_id                     sku_name             category subcategory  \
0  SKU00001     NutriPlus Cookware Large       Home & Kitchen    Cookware   
1  SKU00002  CrispKing Bread Family Pack       Dairy

In [5]:
def normalize_sku_id(series):
    numeric = series.str.replace("SKU", "", regex=False).astype(int)
    return "SKU" + numeric.astype(str).str.zfill(SKU_ID_WIDTH)

def normalize_store_id(series):
    numeric = series.str.replace("ST", "", regex=False).astype(int)
    return "ST" + numeric.astype(str).str.zfill(STORE_ID_WIDTH)

In [6]:
sale = raw["sales"].copy()

before = len(sale)
sale = sale.drop_duplicates(subset="receipt_id")
if before - len(sale):
    note(f"sales: dropped {before - len(sale)} duplicate receipt_id rows")

bad_qty = (sale["customer_quantity"] <= 0).sum()
bad_price = (sale["unit_price"] <= 0).sum()
if bad_qty:
    sale = sale[sale["customer_quantity"] > 0]
    note(f"sales: dropped {bad_qty} rows with non-positive customer_quantity")
if bad_price:
    sale = sale[sale["unit_price"] > 0]
    note(f"sales: dropped {bad_price} rows with non-positive unit_price")

sale["sku_id"] = normalize_sku_id(sale["sku_id"])
sale["store_id"] = normalize_store_id(sale["store_id"])
sale["date"] = pd.to_datetime(sale["date"])
sale["is_promo"] = sale["discount_promo_id"].notna()
note("sales: normalized sku_id (4->5 digit) and store_id (3->2 digit)")

n_transactions = len(sale)
sales_daily = (
    sale.groupby(["date", "sku_id"], as_index=False)
    .agg(
        units_sold=("customer_quantity", "sum"),
        revenue=("total_value", "sum"),
        n_transactions=("receipt_id", "count"),
        promo_flag=("is_promo", "max"),
    )
)
sales_daily["unit_price"] = (sales_daily["revenue"] / sales_daily["units_sold"]).round(2)
note(f"sales: aggregated {n_transactions:,} transactions into {len(sales_daily):,} (date, sku_id) rows")
sales_daily.head()

INFO | sales: normalized sku_id (4->5 digit) and store_id (3->2 digit)
INFO | sales: aggregated 500,000 transactions into 167,973 (date, sku_id) rows


,date,sku_id,units_sold,revenue,n_transactions,promo_flag,unit_price
0,2024-01-01,SKU00001,10,475.00,4,True,47.50
1,2024-01-01,SKU00002,10,6309.28,4,True,630.93
2,2024-01-01,SKU00003,5,2225.23,3,True,445.05
3,2024-01-01,SKU00004,3,653.70,1,False,217.90
4,2024-01-01,SKU00005,17,4282.04,5,True,251.88


In [7]:
cal = raw["calendar"].copy()

before = len(cal)
cal = cal.drop_duplicates(subset="date")
if before - len(cal):
    note(f"calendar: dropped {before - len(cal)} duplicated date rows")

cal["date"] = pd.to_datetime(cal["date"])
SEASON_BY_MONTH = {
    1: "Winter", 2: "Winter", 3: "Winter",
    4: "Spring", 5: "Spring", 6: "Spring",
    7: "Summer", 8: "Summer", 9: "Summer",
    10: "Fall", 11: "Fall", 12: "Fall",
}

extend_to = sales_daily["date"].max()
if extend_to > cal["date"].max():
    gap_start = cal["date"].max() + pd.Timedelta(days=1)
    gap_dates = pd.date_range(gap_start, extend_to, freq="D")
    gap = pd.DataFrame({"date": gap_dates})
    gap["year"] = gap["date"].dt.year
    gap["quarter"] = gap["date"].dt.quarter
    gap["month"] = gap["date"].dt.month
    gap["month_name"] = gap["date"].dt.month_name()
    gap["week"] = gap["date"].dt.isocalendar().week.astype(int)
    gap["day_of_week"] = gap["date"].dt.dayofweek
    gap["day_name"] = gap["date"].dt.day_name()
    gap["is_weekend"] = gap["day_of_week"].isin([5, 6]).astype(int)
    gap["season"] = gap["month"].map(SEASON_BY_MONTH)
    cal = pd.concat([cal, gap], ignore_index=True)
    note(f"calendar: extended by {len(gap_dates)} rows to cover sales through {extend_to:%Y-%m-%d}")

if "is_holiday" not in cal.columns:
    cal["is_holiday"] = False
    note("calendar: no holiday indicator in source - added 'is_holiday' defaulted to False")

if "promo_event" not in cal.columns:
    cal["promo_event"] = pd.NA
    note("calendar: no promo_event column - promo signal comes from sales-level discount_promo_id instead") 

cal.tail()

INFO | calendar: extended by 61 rows to cover sales through 2025-12-31
INFO | calendar: no holiday indicator in source - added 'is_holiday' defaulted to False
INFO | calendar: no promo_event column - promo signal comes from sales-level discount_promo_id instead


,date,year,quarter,month,month_name,week,day_of_week,day_name,is_weekend,season,is_holiday,promo_event
1821,2025-12-27,2025,4,12,December,52,5,Saturday,1,Fall,False,<NA>
1822,2025-12-28,2025,4,12,December,52,6,Sunday,1,Fall,False,<NA>
1823,2025-12-29,2025,4,12,December,1,0,Monday,0,Fall,False,<NA>
1824,2025-12-30,2025,4,12,December,1,1,Tuesday,0,Fall,False,<NA>
1825,2025-12-31,2025,4,12,December,1,2,Wednesday,0,Fall,False,<NA>


In [8]:
sku = raw["sku"].copy()
before = len(sku)
if before - len(sku):
    note(f"sku_master: dropped {before - len(sku)} duplicated sku_id rows")

sku["category"] = sku["category"].str.strip()
sku["subcategory"] = sku["subcategory"].str.strip()

active_sku_ids = set(sales_daily["sku_id"].unique())
total_catalog = len(sku)
sku_master = sku[sku["sku_id"].isin(active_sku_ids)].copy()
sku_master = sku_master.rename(columns={"unit_price": "list_price", "cost_price":  "unit_price"})
note(f"sku_master: catalog has {total_catalog:,} SKUs; filtered to {len(sku_master):,} active (sold) SKUs")
sku_master.head()

INFO | sku_master: catalog has 5,000 SKUs; filtered to 250 active (sold) SKUs


,sku_id,sku_name,category,subcategory,list_price,unit_price,brand
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,SKU00004,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,SKU00005,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [9]:
inv = raw["inventory"].copy()
before = len(inv)
inv = inv.drop_duplicates(subset=["store_id", "sku_id"])
if before - len(inv):
    note(f"inventory: dropped {before - len(inv)} duplicates (store_id, sku_id) rows")

inv["sku_id"] = normalize_sku_id(inv["sku_id"])
inv["store_id"] = normalize_store_id(inv["store_id"])
inv["last_restock_date"] = pd.to_datetime(inv["last_restock_date"])

negative_stock = (inv["stock_on_hand"] < 0).sum()
if negative_stock:
    inv["stock_on_hand"] = inv["stock_on_hand"].clip(lower=0)
    note(f"inventory: clipped {negative_stock} negative stock_on_hand rows to 0")

note("inventory: point-in-time snapshot, not a dated time series - risk scoring will use stock_on_hand, safety_stock, reorder_point")

inventory_position = (
    inv.groupby("sku_id", as_index=False)
    .agg(
        stock_on_hand=("stock_on_hand", "sum"),
        reorder_point=("reorder_point", "sum"),
        safety_stock=("safety_stock", "sum"),
        n_stores_carrying=("store_id", "nunique"),
        last_restock_date=("last_restock_date", "max"),
    )
    .merge(sku_master[["sku_id", "category", "subcategory"]], on="sku_id", how="left")

)
note(f"inventory_position: aggregated to {len(inventory_position):,} SKU-level rows")
inventory_position.head()

INFO | inventory: point-in-time snapshot, not a dated time series - risk scoring will use stock_on_hand, safety_stock, reorder_point
INFO | inventory_position: aggregated to 4,495 SKU-level rows


,sku_id,stock_on_hand,reorder_point,safety_stock,n_stores_carrying,last_restock_date,category,subcategory
0,SKU00001,215,100,33,1,2025-12-22,Home & Kitchen,Cookware
1,SKU00002,0,1582,573,25,2025-10-06,Dairy & Bakery,Bread
2,SKU00003,414,250,85,4,2025-12-13,Stationery & Office,Notebooks
3,SKU00004,148,99,35,1,2025-12-03,Dairy & Bakery,Eggs
4,SKU00005,411,284,104,5,2025-12-26,Home Care,Pest Control


In [10]:
master_dataset = sales_daily.merge(cal, on="date", how="left")
unmatched_dates = master_dataset["season"].isna().sum()
if unmatched_dates:
    note(f"master_dataset: {unmatched_dates} sales rows had no matching calendar date")

master_dataset = master_dataset.merge(sku_master, on="sku_id", how="left")
unmatched_skus = master_dataset["category"].isna().sum()
if unmatched_skus:
    note(f"master_dataset: {unmatched_skus} sales rows had no matching sku_master row")

note(f"master_dataset: {len(master_dataset):,} rows, {len(master_dataset.columns)} columns")
master_dataset.head()

INFO | master_dataset: 167,973 rows, 24 columns


,date,sku_id,units_sold,revenue,n_transactions,promo_flag,unit_price_x,year,quarter,month,...,is_weekend,season,is_holiday,promo_event,sku_name,category,subcategory,list_price,unit_price_y,brand
0,2024-01-01,SKU00001,10,475.00,4,True,47.50,2024,1,1,...,0,Winter,False,<NA>,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,2024-01-01,SKU00002,10,6309.28,4,True,630.93,2024,1,1,...,0,Winter,False,<NA>,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,2024-01-01,SKU00003,5,2225.23,3,True,445.05,2024,1,1,...,0,Winter,False,<NA>,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,2024-01-01,SKU00004,3,653.70,1,False,217.90,2024,1,1,...,0,Winter,False,<NA>,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,2024-01-01,SKU00005,17,4282.04,5,True,251.88,2024,1,1,...,0,Winter,False,<NA>,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [11]:
print("Nulls introduced by joins?")
print(master_dataset.isna().sum()[master_dataset.isna().sum() > 0])
print("\nDate range:", master_dataset["date"].min(), "to", master_dataset["date"].max())
print("Unique SKUs:", master_dataset["sku_id"].nunique())


Nulls introduced by joins?
promo_event    167973
dtype: int64

Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Unique SKUs: 250


In [12]:
sales_daily.to_csv(PROCESSED_DIR / "sales_daily.csv", index=False)
sku_master.to_csv(PROCESSED_DIR / "sku_master.csv", index=False)
cal.to_csv(PROCESSED_DIR / "calendar.csv", index=False)
inventory_position.to_csv(PROCESSED_DIR / "inventory_position.csv", index=False)
master_dataset.to_csv(PROCESSED_DIR / "master_dataset.csv", index=False)

report_lines = ["#FORESIGHT - Data Quality Report", ""] + [f"- {e}" for e in dq_log]
(REPORTS_DIR / "data_quality_report.md").write_text("\n".join(report_lines), encoding="utf-8")
print("Saved.")

Saved.
